# HA0 — 40-sense inventory and annotation review

This notebook presents the team's **20 Chinese + 20 English** spatial-relation inventory and the evidence behind the HA1 input. It reads the four CSV files beside this notebook. Every check is visible here; no project `.py` module is imported. The original lexical inventory was confirmed by Xia Yuqi, while the narrowed EN15 wording is a later editorial paraphrase checked against Oxford Learner's Dictionaries. Editorial decisions do not imply three independent ratings or team sign-off.

## 1. Load and check the 40 selected senses

Each row fixes one term, part of speech, target literal sense, category parent, example, and seven channel scores. The examples are constructed illustrations. The CSV retains source and confidence columns for the ratings.

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

current = Path.cwd().resolve()
def find_input(name):
    for base in (current, *current.parents):
        candidate = base / name
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(f"Cannot find {name}; open this notebook in homework or the repository")

ha0_path = find_input("ha0_zh20_en20.csv")
parent_path = find_input("category_parent_nodes.csv")
audit_path = find_input("feature_score_audit.csv")
final_path = find_input("spatial_relations.csv")
ha0 = pd.read_csv(ha0_path)
parents = pd.read_csv(parent_path)
audit = pd.read_csv(audit_path)
final = pd.read_csv(final_path)
FEATURES = ["visual_0_2", "auditory_0_2", "tactile_0_2", "motor_0_2",
            "proprioceptive_0_2", "interoceptive_0_2", "social_0_2"]
assert len(ha0) == 40 and ha0.concept_id.is_unique
assert ha0.language.value_counts().to_dict() == {"ZH": 20, "EN": 20}
assert ha0[FEATURES].isin([0, 1, 2]).all().all()
assert ha0.category_parent.notna().all()
print("40 senses validated: 20 Chinese, 20 English")
display(ha0[["concept_id", "language", "term", "part_of_speech", "definition_en",
             "category_parent", "example_sentence"]])

40 senses validated: 20 Chinese, 20 English


,concept_id,language,term,part_of_speech,definition_en,category_parent,example_sentence
0,ZH01,ZH,位置,noun,A specified point or extent in a reference space.,spatial_location,传感器的位置标在地图上。
1,ZH02,ZH,内部,noun,The region enclosed by a reference entity.,bounded_region_relation,电池安装在机器人躯干的内部。
2,ZH03,ZH,外部,locative_noun,A region beyond the boundary of a reference en...,bounded_region_relation,摄像头固定在外壳的外部。
3,ZH04,ZH,边界,noun,A line delimiting the extent of an area.,spatial_limit,白线标出了运动区域的边界。
4,ZH05,ZH,中心,noun,A central point fixed by the limits of a refer...,reference_point_relation,圆盘的中心有一个小孔。
5,ZH06,ZH,前面,noun,The part of an entity facing a specified obser...,oriented_location,门的前面装着一盏灯。
6,ZH07,ZH,左侧,locative_noun,A region on the left of a reference entity und...,oriented_location,传感器安装在机身左侧。
7,ZH08,ZH,坡度,noun,The degree to which a surface inclines relativ...,spatial_arrangement,这段斜坡的坡度约为十五度。
8,ZH09,ZH,上方,locative_noun,A region higher than a reference entity along ...,oriented_location,吊灯在桌子的上方。
9,ZH10,ZH,下方,locative_noun,A region lower than a reference entity along t...,oriented_location,电源线从桌子的下方穿过。


## 2. One broader category parent per sense

The eleven pedagogical parent types sit beneath `spatial_or_embodied_relation`. They describe broader relation families; they are **not** the COW/OMW taxonomy paths used to calculate S. The parent definitions explicitly cover geometric properties (for example slope and angle) and openings (for example gaps).

In [2]:
assert len(parents) == 12 and parents.category_parent.is_unique
assert set(ha0.category_parent).issubset(set(parents.category_parent))
display(parents[["category_parent", "definition_en", "supercategory"]])
display(ha0.groupby(["category_parent", "language"]).size().unstack(fill_value=0))

,category_parent,definition_en,supercategory
0,spatial_or_embodied_relation,"A relation involving location, arrangement, ro...",NaN
1,spatial_location,A place specified relative to a reference space.,spatial_or_embodied_relation
2,bounded_region_relation,A location defined by being within or beyond a...,spatial_or_embodied_relation
3,spatial_limit,A spatial divider that marks where a region ends.,spatial_or_embodied_relation
4,reference_point_relation,A position determined by a figure's extent or ...,spatial_or_embodied_relation
5,oriented_location,"A place specified using a direction, axis, or ...",spatial_or_embodied_relation
6,intervening_region_relation,A region located with respect to two bounding ...,spatial_or_embodied_relation
7,spatial_interval_relation,A measurable or perceptible separation or open...,spatial_or_embodied_relation
8,spatial_arrangement,A spatial configuration or geometric property ...,spatial_or_embodied_relation
9,route_or_traversal_relation,"A route or cross-region relation, whether stat...",spatial_or_embodied_relation


language,EN,ZH
category_parent,,
bounded_region_relation,1,2
intervening_region_relation,0,1
load_or_stability_relation,2,1
nearness_or_touch_relation,2,1
oriented_location,2,6
reference_point_relation,1,1
route_or_traversal_relation,3,2
spatial_arrangement,3,2
spatial_interval_relation,4,2


## 3. Seven-feature scores, reasons, and provenance

For the selected literal sense, **0** means no typical direct grounding, **1** conditional or secondary grounding, and **2** characteristic grounding. The seven channels are Visual, Auditory, Tactile, Motor, Proprioceptive, Interoceptive, and Social. Do not add social or philosophical metaphorical readings to a literal spatial score. The audit contains one reason, source, and confidence level for every score. The `manual` label means an earlier single rating pass with no recorded rater identity; it does not establish three-rater agreement.

In [3]:
assert len(audit) == 280
assert audit.groupby("concept_id").size().eq(7).all()
long = ha0.melt(id_vars="concept_id", value_vars=FEATURES,
                var_name="feature", value_name="ha0_score")
checked = long.merge(audit[["concept_id", "feature", "reviewed_score"]],
                     on=["concept_id", "feature"], validate="one_to_one")
assert len(checked) == 280
assert (checked.ha0_score == checked.reviewed_score).all()
print("All 280 HA0 scores match the audit.")
display(audit.score_disagreement_status.value_counts().rename_axis("decision").to_frame("cells"))
display(audit.provenance_label.value_counts().rename_axis("origin").to_frame("cells"))
display(audit.loc[audit.score_disagreement_status != "retained_no_PR17_change_proposed",
                  ["concept_id", "feature", "v2_score", "reviewed_score",
                   "score_disagreement_status", "reason_or_source", "confidence"]])

All 280 HA0 scores match the audit.


,cells
decision,
retained_no_PR17_change_proposed,271
retained_with_explicit_reason_for_HA1,5
adopted_for_HA1_submission,4


,cells
origin,
manual,155
generated,125


,concept_id,feature,v2_score,reviewed_score,score_disagreement_status,reason_or_source,confidence
11,ZH02,proprioceptive_0_2,1,0,adopted_for_HA1_submission,A static interior region does not entail a bod...,medium
18,ZH03,proprioceptive_0_2,1,0,adopted_for_HA1_submission,A static exterior region does not entail a bod...,medium
109,ZH16,proprioceptive_0_2,1,1,retained_with_explicit_reason_for_HA1,A route can conditionally guide egocentric ori...,low
121,ZH18,tactile_0_2,1,1,retained_with_explicit_reason_for_HA1,Tactile contact is conditional in a material r...,low
214,EN11,proprioceptive_0_2,2,2,retained_with_explicit_reason_for_HA1,Body or robot joint-position feedback directly...,medium
221,EN12,proprioceptive_0_2,1,0,adopted_for_HA1_submission,Static containment does not entail a body-posi...,medium
241,EN15,motor_0_2,1,2,adopted_for_HA1_submission,The narrowed crossing sense directly involves ...,medium
242,EN15,proprioceptive_0_2,1,1,retained_with_explicit_reason_for_HA1,Body-relative feedback can conditionally accom...,medium
256,EN17,proprioceptive_0_2,1,1,retained_with_explicit_reason_for_HA1,A route can conditionally guide egocentric ori...,low


## 4. Explicit uncertainty and testable question

**Uncertainty row: ZH18 接触.** Ideal geometric tangency can exist without felt touch, while a physical instance may yield tactile evidence. The selected Tactile score is 1 with low confidence; the CSV records this one-line uncertainty. EN15 *across* was narrowed to the movement sense before HA1 and has Motor=2.

**Question:** In held-out paired spatial scenes, does reversing the observer viewpoint change orientation labels more often than containment labels while the objects and their physical relations stay fixed?

**Prediction:** Orientation labels change more often. **Falsifying result:** After excluding scenes where the physical containment relation changed, the reversal rates are equal or containment labels change more often. The present CSV has no scene-pair observations and does not test this prediction.

In [4]:
display(ha0.loc[ha0.concept_id.isin(["ZH18", "EN15"]),
                ["concept_id", "term", "definition_en", "tactile_0_2", "motor_0_2",
                 "uncertainty_note", "review_status"]])
print("Reported S paths:", final.S_depth.notna().sum(),
      "| S=NA:", final.S_depth.isna().sum())

,concept_id,term,definition_en,tactile_0_2,motor_0_2,uncertainty_note,review_status
17,ZH18,接触,A tangent relation where surfaces meet at a po...,1,0,Geometric tangency can be idealized without ma...,HA1_submission_aligned; team_signoff_not_recorded
34,EN15,across,From one side of a spatial region to the other...,0,2,NaN,HA1_submission_aligned; team_signoff_not_recorded


Reported S paths: 25 | S=NA: 15


## Evidence limits and next step

Four of nine flagged C values were changed for the HA1 input; five were retained with explicit reasons. The 25 inherited S paths have their edge counts checked in the HA1 notebook, but their source synsets have not all been independently verified. Fifteen S values remain NA. `HA_1.ipynb` turns these ratings and paths into the C×S plane, composite A(w), and sensitivity analysis. GitHub publication does not establish LMS submission.